In [ ]:
# ==============================================================================
# 0. CONFIGURACIÓN DEL ENTORNO Y CONEXIÓN S3 (CLOUDFLARE R2)
# Resumen: Inicializa DuckDB con la extensión httpfs para consultar el archivo Parquet
# crudo directamente en Cloudflare R2 sin saturar memoria local.
# ==============================================================================

import os
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

# Configuración estética general de visualizaciones

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

load_dotenv()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
S3_ENDPOINT_URL = os.getenv("S3_ENDPOINT_URL", "").replace("https://", "").rstrip("/")
S3_BUCKET_NAME = os.getenv("S3_BUCKET_NAME")
S3_PREFIX_RAW = os.getenv("S3_PREFIX_RAW", "raw")
AWS_REGION = os.getenv("AWS_DEFAULT_REGION", "eeur")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

con.execute(f"""
    SET s3_endpoint = '{S3_ENDPOINT_URL}';
    SET s3_access_key_id = '{AWS_ACCESS_KEY_ID}';
    SET s3_secret_access_key = '{AWS_SECRET_ACCESS_KEY}';
    SET s3_url_style = 'path';
    SET s3_use_ssl = true;
    SET s3_region = '{AWS_REGION}';
""")

R2_FILE = f"s3://{S3_BUCKET_NAME}/{S3_PREFIX_RAW}/huggingface/fnspid_bancos_raw_completo.parquet"

# ==============================================================================
# 0.1 INSPECCIÓN DE COLUMNAS Y MUESTRA AMPLIA (+10 FILAS) 
# Resumen: Evalúa el esquema físico de la capa Raw y extrae una muestra de 10 filas para auditar la estructura de campos textuales y metadatos.
# ==============================================================================

print("--- ESQUEMA DE COLUMNAS DISPONIBLES EN R2 ---")
schema_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{R2_FILE}');").df()
display(schema_df[['column_name', 'column_type']])

print("\n--- UN REGISTRO POR CADA STOCK_SYMBOL ---")
# Filtramos directamente por el primer registro de cada símbolo en todo el archivo
query_bancos = f"""
    SELECT * 
    FROM read_parquet('{R2_FILE}')
    QUALIFY ROW_NUMBER() OVER (PARTITION BY stock_symbol ORDER BY Date DESC) = 1;
"""
df_por_simbolo = con.execute(query_bancos).df()
display(df_por_simbolo)

In [ ]:
# ==============================================================================
# 0.2 AUDITORÍA DE NULOS Y DETECCIÓN DE COLUMNAS 100% VACÍAS
# Resumen: Evalúa la completitud de cada campo en el dataset íntegro. Identifica columnas que deben ser descartadas en Silver por no aportar información.
# ==============================================================================

print("Cargando dataset completo en memoria para análisis estadístico...")
df = con.execute(f"SELECT * FROM read_parquet('{R2_FILE}');").df()

# Reemplazar representaciones de cadenas vacías por NaN para un conteo riguroso
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)
df.replace(['None', 'nan', 'NULL', 'null'], np.nan, inplace=True)

# Cálculo de valores nulos absolutos y relativos

total_filas = len(df)
nulos_resumen = pd.DataFrame({
    'columna': df.columns,
    'conteo_nulos': df.isnull().sum().values,
    'porcentaje_nulos': (df.isnull().sum().values / total_filas) * 100
}).sort_values(by='porcentaje_nulos', ascending=False).reset_index(drop=True)

print(f"Total de registros analizados: {total_filas:,}")
display(nulos_resumen)

# Identificación explícita de columnas 100% nulas

columnas_100_nulas = nulos_resumen[nulos_resumen['porcentaje_nulos'] >= 99.99]['columna'].tolist()
print("\n" + "="*70)
if columnas_100_nulas:
    print(f"ALERTA: Se detectaron {len(columnas_100_nulas)} columnas 100% NULAS:")
    for col in columnas_100_nulas:
        print(f"  - {col}")
else:
    print("CONFIRMACIÓN: No existen columnas que sean 100% nulas en el dataset.")
print("="*70)

# Visualización: Mapa de calor de valores faltantes (Missing Values)

plt.figure(figsize=(12, 5))
sns.barplot(data=nulos_resumen, x='porcentaje_nulos', y='columna', hue='columna', palette='Reds_r', legend=False)
plt.title('Porcentaje de Valores Nulos o Vacíos por Columna (Total: 41,433 filas)', fontsize=14, weight='bold')
plt.xlabel('% de Nulos')
plt.ylabel('Columna')
plt.xlim(0, 100)
for index, value in enumerate(nulos_resumen['porcentaje_nulos']):
    plt.text(value + 1, index, f"{value:.2f}%", va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# 1. DISTRIBUCIÓN POR ENTIDAD (TICKERS)
# Resumen: Cuantifica el volumen de cobertura mediática de cada uno de los 6 bancos. Permite detectar desbalances de clase antes del entrenamiento del modelo predictivo.
# ==============================================================================

ticker_counts = df['Stock_symbol'].value_counts().reset_index()
ticker_counts.columns = ['Ticker', 'Total_Noticias']
ticker_counts['Porcentaje'] = (ticker_counts['Total_Noticias'] / total_filas) * 100

display(ticker_counts)

plt.figure(figsize=(10, 5))
bars = plt.bar(ticker_counts['Ticker'], ticker_counts['Total_Noticias'], color='#1f77b4', edgecolor='black')
plt.title('Distribución Total de Noticias por Banco (Capa Raw)', fontsize=14, weight='bold')
plt.xlabel('Ticker Bancario')
plt.ylabel('Cantidad de Noticias')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 300, f"{yval:,}", ha='center', va='bottom', weight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# 2. DINÁMICA TEMPORAL Y SINCRONIZACIÓN CON EL MERCADO
# Resumen: Parsea fechas a datetime UTC y analiza la cobertura por año, mes,  día de la semana y distribución horaria para prevenir data leakage y planificar
# el alineamiento bursátil de fines de semana.
# ==============================================================================

# Parseo no destructivo en el dataframe de análisis

df['datetime_utc'] = pd.to_datetime(df['Date'], utc=True, errors='coerce')
df['year'] = df['datetime_utc'].dt.year
df['month'] = df['datetime_utc'].dt.month
df['day_name'] = df['datetime_utc'].dt.day_name()
df['hour'] = df['datetime_utc'].dt.hour

# 2.1 Cobertura Anual por Ticker

noticias_anuales = df.groupby(['year', 'Stock_symbol']).size().unstack(fill_value=0)

plt.figure(figsize=(14, 6))
noticias_anuales.plot(kind='bar', stacked=True, colormap='tab10', ax=plt.gca(), edgecolor='black')
plt.title('Evolución Histórica Anual de Noticias por Banco (2010 - 2023)', fontsize=14, weight='bold')
plt.xlabel('Año')
plt.ylabel('Volumen de Noticias')
plt.legend(title='Banco', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

# 2.2 Ciclo Semanal: Días Hábiles vs Fin de Semana

dias_orden = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
noticias_semanales = df['day_name'].value_counts().reindex(dias_orden)

plt.figure(figsize=(10, 4))
colors = ['#2ca02c' if d not in ['Saturday', 'Sunday'] else '#d62728' for d in dias_orden]
bars = plt.bar(noticias_semanales.index, noticias_semanales.values, color=colors, edgecolor='black')
plt.title('Distribución de Publicaciones por Día de la Semana', fontsize=14, weight='bold')
plt.xlabel('Día de la Semana (Rojo = Fin de Semana / Mercado Cerrado)')
plt.ylabel('Total Noticias')
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2.0, bar.get_height() + 150, f"{bar.get_height():,}", ha='center', va='bottom')
plt.tight_layout()
plt.show()

In [ ]:
# 1. Asegurar la columna de año (detectando el nombre exacto de la columna de símbolos)
col_simbolo = 'stock_symbol' if 'stock_symbol' in df.columns else 'Stock_symbol'
df['datetime_utc'] = pd.to_datetime(df['Date'], utc=True, errors='coerce')
df['year'] = df['datetime_utc'].dt.year

# 2. Matriz completa de recuentos anuales por banco
disponibilidad = df.groupby(['year', col_simbolo]).size().unstack(fill_value=0)

print("=" * 80)
print("MATRIZ DE RECUENTO ANUAL DE NOTICIAS POR BANCO")
print("=" * 80)
display(disponibilidad)

# 3. Resumen analítico por banco (Años con cobertura, años vacíos y volumen total)
resumen_cobertura = []
todos_los_anios = set(disponibilidad.index)

for banco in disponibilidad.columns:
    serie = disponibilidad[banco]
    anios_con_datos = serie[serie > 0].index.tolist()
    anios_sin_datos = sorted(list(todos_los_anios - set(anios_con_datos)))
    total_noticias = serie.sum()
    
    resumen_cobertura.append({
        'Banco': banco,
        'Total Noticias': total_noticias,
        'Primer Año': min(anios_con_datos) if anios_con_datos else None,
        'Último Año': max(anios_con_datos) if anios_con_datos else None,
        'Años Cubiertos': len(anios_con_datos),
        'Años con 0 Noticias': anios_sin_datos if anios_sin_datos else 'Ninguno'
    })

df_resumen = pd.DataFrame(resumen_cobertura).sort_values(by='Total Noticias', ascending=False)

print("\n" + "=" * 80)
print("DIAGNÓSTICO DE COBERTURA Y AÑOS FALTANTES")
print("=" * 80)
display(df_resumen)

In [ ]:
# ==============================================================================
# 3. CALIDAD TEXTUAL Y LONGITUD (CRITERIO TRANSFORMERS / FINBERT)
# Resumen: Mide la longitud en palabras de titulares, resúmenes y cuerpo completo. Orienta la decisión entre truncamiento de artículos o uso de resúmenes extractivos 
# frente al límite de 512 tokens de modelos de lenguaje.
# ==============================================================================

def count_words(text_series):
    return text_series.fillna('').astype(str).apply(lambda x: len(x.split()) if x.strip() else 0)

df['words_title'] = count_words(df['Article_title'])
df['words_lsa'] = count_words(df['Lsa_summary'])
df['words_textrank'] = count_words(df['Textrank_summary'])
df['words_article'] = count_words(df['Article'])

longitudes_resumen = df[['words_title', 'words_lsa', 'words_textrank', 'words_article']].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).T[['count', 'mean', 'std', 'min', '50%', '90%', '95%', 'max']]

display(longitudes_resumen)

# Visualización comparativa de distribuciones

fig, axes = plt.subplots(1, 3, figsize=(10, 5))

# Título

sns.histplot(df['words_title'], bins=40, kde=True, ax=axes[0], color='#1f77b4')
axes[0].set_title('Longitud Titular (Article_title)')
axes[0].set_xlabel('Nº de Palabras')
axes[0].axvline(df['words_title'].median(), color='red', linestyle='--', label=f'Mediana: {df["words_title"].median():.0f}')
axes[0].legend()

# Resumen TextRank

sns.histplot(df[df['words_textrank'] > 0]['words_textrank'], bins=40, kde=True, ax=axes[1], color='#ff7f0e')
axes[1].set_title('Longitud TextRank Summary')
axes[1].set_xlabel('Nº de Palabras')
axes[1].axvline(df[df['words_textrank'] > 0]['words_textrank'].median(), color='red', linestyle='--', label=f'Mediana: {df[df["words_textrank"] > 0]["words_textrank"].median():.0f}')
axes[1].legend()

# Artículo Completo (Acotado a percentil 95 para visualización)

p95_art = df['words_article'].quantile(0.95)
sns.histplot(df[(df['words_article'] > 0) & (df['words_article'] <= p95_art)]['words_article'], bins=40, kde=True, ax=axes[2], color='#2ca02c')
axes[2].set_title(f'Longitud Artículo Completo (≤ P95: {p95_art:.0f} palabras)')
axes[2].set_xlabel('Nº de Palabras')
axes[2].axvline(512, color='black', linestyle=':', label='Límite Típico BERT (~512)')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# 4. DUPLICIDAD Y CONCENTRACIÓN EDITORIAL
# Resumen: Cuantifica noticias redundantes (reemisiones) que sesgarían el sentimiento, y evalúa la concentración de medios (Publisher) para filtrar ruido o sindicación masiva.
# ==============================================================================

# 4.1 Análisis de Duplicados

titulos_duplicados = df['Article_title'].duplicated().sum()
titulos_unicos = df['Article_title'].nunique()
combinacion_duplicada = df.duplicated(subset=['Date', 'Stock_symbol', 'Article_title']).sum()

print("--- AUDITORÍA DE REDUNDANCIA Y DUPLICADOS ---")
print(f"Titulares únicos: {titulos_unicos:,} de {total_filas:,}")
print(f"Titulares repetidos en el dataset: {titulos_duplicados:,} ({(titulos_duplicados/total_filas)*100:.2f}%)")
print(f"Duplicados exactos en (Fecha + Ticker + Titular): {combinacion_duplicada:,} ({(combinacion_duplicada/total_filas)*100:.2f}%)")

# 4.2 Top 10 Fuentes (Publishers)

top_publishers = df['Publisher'].fillna('Desconocido').value_counts().head(10).reset_index()
top_publishers.columns = ['Publisher', 'Total_Artículos']

plt.figure(figsize=(12, 5))
sns.barplot(
    data=top_publishers, 
    x='Total_Artículos', 
    y='Publisher', 
    hue='Publisher', 
    palette='viridis', 
    legend=False
)
plt.title('Top 10 Publishers con Mayor Volumen de Cobertura', fontsize=14, weight='bold')
plt.xlabel('Cantidad de Noticias')
plt.ylabel('Medio de Comunicación')
for index, value in enumerate(top_publishers['Total_Artículos']):
    plt.text(value + 100, index, f"{value:,}", va='center', weight='bold', fontsize=9)
plt.tight_layout()
plt.show()